## Before You Begin

1. Install and enable the suggested extensions:
   - Python (Microsoft)
   - Jupyter (Microsoft)
2. Wait for both extensions to finish installing.

> **Note:** If the notebook does not run, no kernel appears, or you encounter a Jupyter activation error:
>
> - Open the **Jupyter** extension.
> - Click the **▼** next to **Uninstall**.
> - Select **Install Specific Version...**
> - Install **Jupyter v2025.4.0**.
> - Reload VS Code and reopen the notebook.

## Configure the Kernel

1. Click **Select Kernel**.
2. Under **Kernel Source**, select **Python Environments...**
3. Choose **Python 3.12.1**.

You can now run the notebook.

> **Tip:** Keep the Azure AI Foundry project page open in Microsoft Edge for easy access during the lab. If multiple AI Foundry resources are available, confirm with the instructor which resource should be used. The resource and model deployed during the demonstration can be used unless instructed otherwise.

# Calling the model

Now, this notebook lets you:

1. Install the Azure AI Inference SDK
2. Configure endpoint credentials
3. Send chat requests (non‑streaming & streaming)
4. Run small prompt‑tuning experiments


In [ ]:
# Install packages
%pip install --upgrade openai requests

## Configure your endpoint
Fill in the `ENDPOINT`, `API_KEY`, and `DEPLOYMENT` values from your Azure AI Foundry deployment page.

**Ensure to trim the `Target URI` according to the format given below**


In [ ]:
ENDPOINT = "https://hakunamatata1234.services.ai.azure.com/openai/v1"
API_KEY = "<REPLACE_WITH_YOUR_API_KEY>"  # Foundry portal → your resource → Keys and Endpoint
DEPLOYMENT = "gpt-5.4-mini"

assert ENDPOINT.startswith("https://"), "Please set a valid ENDPOINT"
assert API_KEY and API_KEY != "<REPLACE_WITH_YOUR_API_KEY>", "Please set your API_KEY"
assert DEPLOYMENT, "Please set your DEPLOYMENT name"

## Initialise a client

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=ENDPOINT, api_key=API_KEY)

## Invoke a basic completion response

In [ ]:
response = client.responses.create(
    model=DEPLOYMENT,
    input=[
        {"role": "system", "content": "You are a concise Azure assistant."},
        {"role": "user",   "content": "In one sentence, define a foundation model."}
    ],
    max_output_tokens=4096,
)
print(response.output_text)

## Streaming example
Set `stream=True` to iterate tokens as they arrive.


In [ ]:
stream = client.responses.create(
    model=DEPLOYMENT,
    input=[
        {"role": "system", "content": "You are a succinct assistant."},
        {"role": "user",   "content": "List 3 prompt tuning levers with a one‑liner each."}
    ],
    max_output_tokens=4096,
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")

# Re-initialising client for rest of the lab

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=ENDPOINT, api_key=API_KEY)

## JSON‑shaped responses
Try to enforce a JSON‑only reply via the system message. If your model supports it, you can also pass a `response_format`. If not, validate JSON client‑side.


In [ ]:
import json

messages_input = [
    {"role": "system", "content": "Return ONLY valid JSON with keys: summary (string), bullets (array)."},
    {"role": "user",   "content": "Summarize what the Chat playground is used for and give 3 bullets."}
]
response = client.responses.create(
    model=DEPLOYMENT,
    input=messages_input,
    text={"format": {"type": "json_object"}},
    max_output_tokens=4096,
)
text = response.output_text
print(text)

try:
    obj = json.loads(text)
    print("\nParsed JSON keys:", list(obj.keys()))
except Exception as e:
    print("\nJSON parse failed:", e)

## Parameter sweep for model optimisation: temperature, max_tokens & top_p
Run a small grid to see how outputs change.


In [ ]:
prompt = "Explain embeddings vs. fine‑tuning for short‑form classification in 3 bullets."
for effort in ["low", "medium", "high"]:
    for mot in [200, 400]:
        res = client.responses.create(
            model=DEPLOYMENT,
            input=[{"role": "user", "content": prompt}],
            reasoning={"effort": effort},
            max_output_tokens=mot,
        )
        print(f"\n--- reasoning_effort={effort} max_output_tokens={mot}")
        print(res.output_text)

## REST via `requests`
You can also call the `/chat/completions` route directly.


In [ ]:
import requests, json

url = f"{ENDPOINT}/responses"
headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
payload = {
    "model": DEPLOYMENT,
    "input": [
        {"role": "system", "content": "You are a concise Azure assistant."},
        {"role": "user",   "content": "Name two cases to prefer a serverless API deployment."}
    ],
    "max_output_tokens": 250
}
resp = requests.post(url, headers=headers, json=payload, timeout=60)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2)[:2000])

# Conclusion & Next Steps

## What you accomplished

- Navigated Azure AI Foundry → explored Model catalog → reviewed model pages.

- Deployed a chat-capable foundation model and verified endpoint details.

- Tested prompts in Chat Playground (system message, safety, parameters).

- Called the model via SDK/REST and ran small tuning experiments (temperature, top-p, max tokens).